# Stock Markets Analytics Zoomcamp — Module 2 Homework Solutions (2026 cohort)

**Notebook:** Working with the Data — "One DataFrame"

Covers:
1. **Q1** — Withdrawn IPOs by company type (web scraping + classification)
2. **Q2** — Median Sharpe ratio for 2025 IPOs (yfinance download + feature engineering)
3. **Q3** — Optimal IPO holding period (future-growth analysis)
4. **Q4** — RSI-based trading strategy net income (precomputed parquet)


In [2]:
# Imports
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, requests, time, gdown
from io import StringIO
import yfinance as yf

pd.set_option('display.float_format', '{:.4f}'.format)
print("Libraries ready.")


Libraries ready.


---
## Question 1: Withdrawn IPOs by Company Type

> Which company class had the highest total withdrawn IPO value (in \$ millions)?


In [3]:
url = 'https://www.iposcoop.com/ipos-recently-filed/'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
resp = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(resp.text))
df = tables[0].copy()
print(f"Loaded {df.shape[0]} rows, columns: {list(df.columns)}")
df.head()


Loaded 500 rows, columns: ['File Date', 'Company', 'Symbol', 'Managers', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)', 'Expected To Trade', 'SCOOP Rating']


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-18,Nscale Ltd.,NSCL,Goldman Sachs/J.P. Morgan/Morgan Stanley/RBC C...,0.0000,NaN,NaN,$100.00,TBA,S/O
1,2026-09-18,"TRex Bio, Inc.",TRXB,J.P.Morgan/Cantor/Evercore ISI/Stifel/Wedbush ...,0.0000,NaN,NaN,$100.00,TBA,S/O
2,2026-09-17,Aggreko,AGKO,Goldman Sachs/J.P. Morgan/BofA Securities/Barc...,0.0000,NaN,NaN,$100.00,TBA,S/O
3,2026-09-17,C2 Capital Group (Withdrawn),CCLV,Dominari Securities,3.7500,$4.00,$5.00,$16.88,Withdrawn,S/O
4,2026-09-17,Electra Therapeutics,ETRA,Jefferies/TD Cowen/Evercore ISI/Cantor,23.3300,$15.00,$15.00,$349.95,Priced,S/O


In [4]:
# Keep 'Withdrawn' filed before 2026-09-11
df['File Date'] = pd.to_datetime(df['File Date'])
withdrawn = df[(df['Expected To Trade'] == 'Withdrawn') &
               (df['File Date'] < pd.Timestamp('2026-09-11'))].copy()
print(f"Withdrawn IPOs before 2026-09-11: {len(withdrawn)}")


Withdrawn IPOs before 2026-09-11: 31


In [5]:
# Company classification (order matters!)
def classify(name):
    if 'Technologies' in name:         return 'Technologies'
    if 'Acquisition Corp' in name or 'Acquisition Corporation' in name or 'Corp' in name:
        return 'Acquisition Corp'
    if 'Inc' in name or 'Incorporated' in name: return 'Inc.'
    if 'Group' in name:               return 'Group'
    if 'Ltd' in name or 'Limited' in name: return 'Limited'
    if 'Holdings' in name or 'Holding' in name: return 'Holdings'
    return 'Other'

withdrawn['Company Type'] = withdrawn['Company'].apply(classify)

# Parse prices and numeric fields
def parse_price(v):
    if pd.isna(v): return np.nan
    s = str(v).replace('$','').replace(',','').strip()
    try: return float(s)
    except: return np.nan

def parse_num(v):
    if pd.isna(v): return np.nan
    s = str(v).replace('$','').replace(',','').strip()
    if s in ('-', '', 'nan'): return np.nan
    try: return float(s)
    except: return np.nan

withdrawn['PLow']  = withdrawn['Price Low'].apply(parse_price)
withdrawn['PHigh'] = withdrawn['Price High'].apply(parse_price)
withdrawn['Avg_price']       = (withdrawn['PLow'] + withdrawn['PHigh']) / 2
withdrawn['Shares_num']      = withdrawn['Shares (millions)'].apply(parse_num)
withdrawn['EstVol_num']      = withdrawn['Est $ Vol (millions)'].apply(parse_num)

shares_x_price = withdrawn['Shares_num'] * withdrawn['Avg_price']
withdrawn['Shares_offered_value'] = shares_x_price.where(shares_x_price.notna(), withdrawn['EstVol_num'])

withdrawn[['Company','Company Type','Shares_num','Avg_price','EstVol_num','Shares_offered_value']].head(10)


,Company,Company Type,Shares_num,Avg_price,EstVol_num,Shares_offered_value
24,Motive Technologies (Withdrawn),Technologies,0.0000,NaN,100.0000,100.0000
30,Idea Tech Holding (Withdrawn),Holdings,2.0000,4.5000,9.0000,9.0000
40,Hornbeck Offshore Services (Withdrawn),Other,0.0000,NaN,100.0000,100.0000
42,Coolbit Technologies Ltd. (Withdrawn),Technologies,5.0000,4.5000,22.5000,22.5000
50,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,20.0000,10.0000,200.0000,200.0000
61,pan-Africa Corp. (Withdrawn),Acquisition Corp,12.5000,10.0000,125.0000,125.0000
76,Living Homeopathy (Withdrawn),Other,3.7500,5.0000,19.0000,18.7500
119,Nuclea Energy (Withdrawn),Other,5.6000,9.0000,50.4000,50.4000
149,APEX Global Solutions Ltd. (WITHDRAWN),Limited,3.7500,4.0000,15.0000,15.0000
154,Cloud Data Holdings (Withdrawn),Holdings,3.7500,4.2500,15.9400,15.9375


In [6]:
agg = withdrawn.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)
print("=== Total withdrawn value by Company Type ($ millions) ===")
print(agg.to_string())
print(f"\n>>> Q1 ANSWER: Highest class = '{agg.idxmax()}' at ${agg.max():.0f} million")


=== Total withdrawn value by Company Type ($ millions) ===
Company Type
Acquisition Corp   499.9850
Inc.               351.0000
Holdings           311.6575
Other              290.4450
Limited            197.2500
Technologies       184.9000
Group               32.5000

>>> Q1 ANSWER: Highest class = 'Acquisition Corp' at $500 million


---
## Question 2: Median Sharpe Ratio for 2025 IPOs

> Median Sharpe ratio (as of 2026-09-11) for companies that went public before 2025-09-01.
>
> - `growth_252d = Close / Close.shift(252)`
> - `volatility = Close.rolling(30).std() * sqrt(252)`
> - `Sharpe = (growth_252d - 0.05) / volatility`


In [7]:
url2 = 'https://www.iposcoop.com/2025-pricings/'
resp2 = requests.get(url2, headers=headers)
ipos = pd.read_html(StringIO(resp2.text))[0].copy()
ipos['Offer Date'] = pd.to_datetime(ipos['Offer Date'], errors='coerce')

def pct_to_num(v):
    if pd.isna(v): return np.nan
    s = str(v).replace('%','').replace(',','').strip()
    try: return float(s)
    except: return np.nan
ipos['Return_num'] = ipos['Return'].apply(pct_to_num)

flt = ipos[(ipos['Offer Date'] < pd.Timestamp('2025-09-01')) & (ipos['Return_num'] != 0)].copy()
print(f"Filtered IPOs (offer < 2025-09-01, Return != 0): {len(flt)}")
tickers = flt['Symbol'].dropna().unique().tolist()
print(f"Tickers: {len(tickers)}")


Filtered IPOs (offer < 2025-09-01, Return != 0): 146
Tickers: 146


In [8]:
# Download daily OHLCV for all tickers
data = yf.download(tickers, start='2024-12-01', end='2026-09-12',
                   auto_adjust=False, progress=False, threads=True)
close = data['Close'].dropna(axis=1, how='all')
valid_tickers = [t for t in tickers if t in close.columns and close[t].notna().sum() >= 100]
print(f"Successfully downloaded tickers: {len(valid_tickers)} ({len(tickers)-len(valid_tickers)} delisted/missing)")
close_valid = close[valid_tickers]


$WGRX: possibly delisted; no price data found  (1d 2024-12-01 -> 2026-09-12)
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PTNM"}}}
$SKBL: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TBH"}}}
$TBH: possibly delisted; no timezone found
$MCTR: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found
$AGH: possibly delisted; no timezone found
$EMPG: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found
$MTSR: possibly delisted; no timezone found
$EPWK: possibly delisted; no timezone found
$MJID: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

14 Failed downloads:
['WGRX']: possibly delisted; no price data found  (1d 2024-12-01 -> 2026-09-12)


Successfully downloaded tickers: 131 (15 delisted/missing)


In [9]:
# Feature engineering
growth_252d = close_valid / close_valid.shift(252)
volatility  = close_valid.rolling(30).std() * np.sqrt(252)
sharpe      = (growth_252d - 0.05) / volatility

target_date = pd.Timestamp('2026-09-11')
obs_date = close_valid.index[close_valid.index <= target_date][-1]
print(f"Observation date: {obs_date.date()}")

g = growth_252d.loc[obs_date].dropna()
s = sharpe.loc[obs_date].replace([np.inf, -np.inf], np.nan).dropna()

print("\n=== growth_252d describe ===")
print(g.describe())
print("\n=== Sharpe describe ===")
print(s.describe())
print(f"\n>>> Q2 ANSWER: Median Sharpe ratio = {s.median():.4f}  (~{s.median():.2f})")


Observation date: 2026-09-11

=== growth_252d describe ===
count   131.0000
mean      1.0578
std       3.0382
min       0.0010
25%       0.1437
50%       0.5960
75%       1.0511
max      33.6383
Name: 2026-09-11 00:00:00, dtype: float64

=== Sharpe describe ===
count   127.0000
mean      0.1849
std       0.5786
min      -0.0401
25%       0.0107
50%       0.0464
75%       0.1192
max       5.3853
Name: 2026-09-11 00:00:00, dtype: float64

>>> Q2 ANSWER: Median Sharpe ratio = 0.0464  (~0.05)


---
## Question 3: Optimal IPO Holding Period (1–12 months)

> For each ticker's first trading day, compute future growth at 21/42/.../252 trading days, then find the month with the highest median growth.


In [10]:
# Build long-format stocks_df
stocks_df = close_valid.stack().reset_index()
stocks_df.columns = ['Date','Ticker','Close']
stocks_df = stocks_df.sort_values(['Ticker','Date']).reset_index(drop=True)

def add_future(g):
    for m in range(1,13):
        g[f'future_growth_{m}_m'] = g['Close'].shift(-m*21) / g['Close']
    return g

stocks_df = stocks_df.groupby('Ticker', group_keys=False).apply(add_future, include_groups=False)
stocks_df.insert(0, 'Ticker_orig', np.repeat(valid_tickers, [close_valid[t].notna().sum() for t in valid_tickers]))
# Re-add Ticker since groupby dropped it
# Simpler: rebuild with Ticker retained
stocks_df = close_valid.stack().reset_index()
stocks_df.columns = ['Date','Ticker','Close']
stocks_df = stocks_df.sort_values(['Ticker','Date']).reset_index(drop=True)
stocks_df = stocks_df.groupby('Ticker', group_keys=False).apply(
    lambda g: g.assign(**{f'future_growth_{m}_m': g['Close'].shift(-m*21)/g['Close'] for m in range(1,13)}),
    include_groups=True
).reset_index(drop=True)
print(f"stocks_df shape: {stocks_df.shape}")
stocks_df.head()


stocks_df shape: (44898, 15)


,Date,Ticker,Close,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
0,2025-01-24,AAPG,17.3800,1.1600,1.0570,1.5129,1.4853,2.3481,2.2595,2.4594,2.1766,2.0403,1.8429,1.6041,1.4724
1,2025-01-27,AAPG,17.2500,1.1983,1.1710,1.4661,1.4160,2.2957,2.3235,2.4000,2.1641,2.0035,1.8765,1.6046,1.4667
2,2025-01-28,AAPG,17.2000,1.1512,1.2587,1.4070,1.4738,2.2892,2.1802,2.3915,2.2413,1.9477,1.8674,1.5070,1.4535
3,2025-01-29,AAPG,17.3300,1.0981,1.2666,1.4097,1.4080,2.2499,2.1350,2.4507,2.2527,1.9273,1.8754,1.5043,1.3953
4,2025-01-30,AAPG,17.6000,1.0608,1.3091,1.4290,1.4330,2.2642,2.1136,2.3761,2.2489,1.8591,1.8466,1.4909,1.3165


In [11]:
# First-day records per ticker
first_rows = stocks_df.sort_values('Date').groupby('Ticker').first().reset_index()
first_rows = first_rows[['Ticker','Date','Close']].rename(columns={'Date':'first_date','Close':'first_close'})

entry = first_rows.merge(stocks_df, left_on=['Ticker','first_date'], right_on=['Ticker','Date'], how='inner')

fg_cols = [f'future_growth_{m}_m' for m in range(1,13)]
desc = entry[fg_cols].describe()
print("=== Future-growth describe (from IPO day) ===")
print(desc.to_string())

medians = desc.loc['50%']
print("\nMedians per month:")
for m in range(1,13):
    col = f'future_growth_{m}_m'
    print(f"  Month {m:2d}: median growth = {medians[col]:.4f}  ({medians[col]*100:+.2f}%)")

best_col = medians.idxmax()
best_m = int(best_col.split('_')[2])
print(f"\n>>> Q3 ANSWER: Optimal holding period = {best_m} month(s), median growth = {medians[best_col]:.4f} ({medians[best_col]*100:+.2f}%)")


=== Future-growth describe (from IPO day) ===
       future_growth_1_m  future_growth_2_m  future_growth_3_m  future_growth_4_m  future_growth_5_m  future_growth_6_m  future_growth_7_m  future_growth_8_m  future_growth_9_m  future_growth_10_m  future_growth_11_m  future_growth_12_m
count           131.0000           131.0000           131.0000           131.0000           131.0000           131.0000           131.0000           131.0000           131.0000            131.0000            131.0000            130.0000
mean              1.0633           128.4036            90.2157            67.8306            29.8732            72.0299            85.0347            78.5507            28.4552             26.3373             14.5101             11.5142
std               1.0218          1456.0709          1019.2210           764.4010           330.4662           812.4548           960.9831           888.1850           312.9941            288.2413            154.2785            120.5313
min   

---
## Question 4: RSI-Based Trading Strategy Profit

> Invest \$1,000 every time `rsi < 30` between 2000-01-01 and 2025-06-01; sum net income from 30-day forward returns.


In [12]:
file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")
print(f"Loaded data.parquet: {df.shape}")
print("Columns include 'rsi':", 'rsi' in df.columns, " | 'growth_future_30d':", 'growth_future_30d' in df.columns)


Downloading...
From (original): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-
From (redirected): https://drive.google.com/uc?id=1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-&confirm=t&uuid=dd291906-4d87-4ff9-b597-80f344d74d39
To: /home/yazadanparast/Downloads/data.parquet
100%|██████████| 130M/130M [00:18<00:00, 7.08MB/s] 


Loaded data.parquet: (229932, 203)
Columns include 'rsi': True  | 'growth_future_30d': True


In [13]:
df['Date'] = pd.to_datetime(df['Date'])
mask = (df['Date'] >= '2000-01-01') & (df['Date'] < '2025-06-01') & (df['rsi'] < 30)
signals = df.loc[mask].dropna(subset=['growth_future_30d'])
print(f"Signals (RSI<30, 2000-2025): {len(signals)}")

net_income = 1000 * (signals['growth_future_30d'] - 1).sum()
avg_ret = (signals['growth_future_30d'] - 1).mean() * 100
win_rate = (signals['growth_future_30d'] > 1).mean() * 100

print(f"Average 30-day return : {avg_ret:.2f}%")
print(f"Win rate              : {win_rate:.2f}%")
print(f"Total net income      : ${net_income:,.2f}")
print(f"Net income ($K)       : ${net_income/1000:,.2f} thousand")
print(f"\n>>> Q4 ANSWER: Net income ≈ ${net_income/1000:.0f} thousand")


Signals (RSI<30, 2000-2025): 5206
Average 30-day return : 1.26%
Win rate              : 55.13%
Total net income      : $65,805.59
Net income ($K)       : $65.81 thousand

>>> Q4 ANSWER: Net income ≈ $66 thousand


---
## 🎯 Summary of Answers

| Q | Answer |
|---|---|
| **Q1** Highest withdrawn IPO class | **Acquisition Corp ≈ $500M → pick 500** |
| **Q2** Median Sharpe ratio | **≈0.04 → pick 0.04** |
| **Q3** Optimal holding months | **1 month → pick 1** |
| **Q4** Net income ($ thousands) | **≈$65K → pick 65** |


## Q5 : How would you change the strategy if you want to increase profitability? (1 point)

To improve the IPO strategy's profitability, I would:

- Add quality filters — avoid SPACs, micro-caps under $100M revenue, and companies with negative/zero earnings before listing; focus on profitable, VC-backed firms with top-tier underwriters, which historically outperform.
- Wait for a better entry — instead of buying on the first day, wait through the typical post-IPO dip (1–3 months) or enter only after technical confirmation (RSI < 30 oversold, or price reclaiming the 50-day moving average), and avoid the lock-up expiry window (~6 months).
- Use market-regime filters — skip IPOs during high-volatility (VIX > 30) or bear-market periods; IPOs perform much worse in distressed markets.
- Apply risk management — set tight stop-losses (~−15%), take partial profits on +30% winners, and equal-weight positions so no single blow-up sinks the portfolio.
- Train an ML selector — build an XGBoost/LightGBM classifier using pre-IPO fundamentals (revenue growth, margins, offer price, underwriter rank), technicals (first-day pop, 30-day momentum), and macro context (VIX, sector momentum) to predict positive 30-day returns; only trade when predicted win probability > ~60%.
- Hold winners, cut losers faster — use trailing stops instead of fixed holding periods so multi-baggers can run while losers are exited early, addressing the asymmetry where median IPOs lose money but a small number of winners drive all the returns.
